# LightGBM (Tapia et al., 2025): versión para Google Colab

Esta es la sección de LightGBM del código de los autores, con tres arreglos para que corra en Colab:

1. **Error `verbose`:** `LGBMRegressor.fit(..., verbose=False)` fue eliminado en LightGBM 4. Se quitó; el silencio ya lo da `'verbose': -1` en los hiperparámetros.
2. **Tiempo de entrenamiento:** DART con 8.000 iteraciones tarda horas en el procesador de Colab. Hay dos modos: `rapido` (GBDT, 400 iteraciones, unos minutos) y `completo` (hiperparámetros originales de los autores).
3. **Gráficos de cascada:** la clase `ShapObject` del original ya no funciona con las versiones recientes de `shap`; se usa `shap.Explanation`.
4. **Tiempo de SHAP:** el original usa `interventional` con todo el conjunto de entrenamiento (unos 41.000 filas) como fondo y lo calcula sobre esas mismas filas, lo que es inviable. Aquí se calcula sobre una muestra y con un fondo pequeño.

Ejecute las celdas en orden. Elija el modo y la fuente de datos en la celda de configuración.

In [ ]:
# Instalación (Colab ya trae lightgbm, scikit-learn, pandas y matplotlib)
!pip -q install shap
import lightgbm, sklearn, shap
print('lightgbm', lightgbm.__version__, '| sklearn', sklearn.__version__, '| shap', shap.__version__)

## Configuración

In [ ]:
# ---- Elija aquí ----
MODO = 'rapido'          # 'rapido' (minutos) o 'completo' (hiperparámetros originales, horas)
# Datos reales de Mendeley (data_paper_plosone.csv o data_paper_plosone_set.csv; ambos sirven). En el repositorio: data/real/
RUTAS_REAL = ['data_paper_plosone.csv', 'data_paper_plosone_set.csv', '../../data/real/data_paper_plosone_set.csv']
N_MUESTRA_SHAP = 2000    # filas sobre las que se calcula SHAP (en 'completo' use 500 a 1000)
N_FONDO_SHAP = 100       # filas de fondo (solo SHAP interventional)
SEMILLA_CV, SEMILLA_HOLDOUT = 123, 30   # las de los autores
# --------------------
assert MODO in ('rapido', 'completo')

## Carga de datos
Si el archivo no está en Colab, se abre el cuadro para subirlo (también puede montar Google Drive).

In [ ]:
import os, numpy as np, pandas as pd

ruta = next((r for r in RUTAS_REAL if os.path.exists(r)), None)
if ruta is None:
    from google.colab import files
    print('No se encontró el archivo de datos. Suba data_paper_plosone.csv o data_paper_plosone_set.csv:')
    subido = files.upload()
    ruta = list(subido.keys())[0]

properties = pd.read_csv(ruta)
print(properties.shape)
properties.head(3)

## Definición de métricas y de la clase de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import r2_score, make_scorer
from sklearn.model_selection import train_test_split, KFold, cross_validate

# Métricas en UF: se deshace log1p con expm1 (igual que el original)
def mape_metric(y_true, y_pred):
    y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true) * 100)

def mdape_metric(y_true, y_pred):
    y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true) * 100)

def mae_metric(y_true, y_pred):
    y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def rmse_metric(y_true, y_pred):
    y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2_metric(y_true, y_pred):
    return r2_score(np.expm1(y_true), np.expm1(y_pred))


def hiperparametros(modo):
    """'completo' = valores de los autores (DART). 'rapido' = GBDT con 400 iteraciones."""
    comunes = {'objective': 'regression', 'learning_rate': 0.1, 'feature_fraction': 0.9,
               'bagging_fraction': 0.7, 'bagging_freq': 10, 'verbose': -1, 'n_jobs': -1,
               'max_depth': 40, 'num_leaves': 256, 'max_bin': 512, 'min_data_in_leaf': 10}
    if modo == 'completo':
        return dict(comunes, boosting_type='dart', n_estimators=8000, drop_rate=0.12)
    return dict(comunes, boosting_type='gbdt', n_estimators=400)


class Training:
    def __init__(self, X, y, modo):
        self.X, self.y, self.modo = X, y, modo

    def train_cross_validation(self):
        modelo = lgb.LGBMRegressor(**hiperparametros(self.modo))
        kf = KFold(n_splits=5, shuffle=True, random_state=SEMILLA_CV)
        scoring = {'MAPE': make_scorer(mape_metric, greater_is_better=False),
                   'MDAPE': make_scorer(mdape_metric, greater_is_better=False),
                   'MAE': make_scorer(mae_metric, greater_is_better=False),
                   'RMSE': make_scorer(rmse_metric, greater_is_better=False),
                   'R2': make_scorer(r2_metric, greater_is_better=True)}
        # n_jobs=1 en la CV: LightGBM ya usa todos los núcleos dentro de cada ajuste
        cv = cross_validate(modelo, self.X, self.y, cv=kf, scoring=scoring, n_jobs=1)
        return {'MAPE': -cv['test_MAPE'].mean(), 'MDAPE': -cv['test_MDAPE'].mean(),
                'MAE': -cv['test_MAE'].mean(), 'RMSE': -cv['test_RMSE'].mean(),
                'R2': cv['test_R2'].mean()}

    def train_simple_validation(self):
        X_train, X_test, y_train, y_test = train_test_split(self.X, self.y, test_size=0.2,
                                                            random_state=SEMILLA_HOLDOUT)
        # Sin eval_set ni verbose: en LightGBM 4 el parámetro verbose de fit() ya no existe
        modelo = lgb.LGBMRegressor(**hiperparametros(self.modo)).fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        metricas = {'MAPE': mape_metric(y_test, y_pred), 'MDAPE': mdape_metric(y_test, y_pred),
                    'MAE': mae_metric(y_test, y_pred), 'RMSE': rmse_metric(y_test, y_pred),
                    'R2': r2_metric(y_test, y_pred)}
        return modelo, metricas, X_train, X_test, y_train, y_test, y_pred

## Entrenamiento y validación cruzada (5 particiones)
Modelo final con las 34 variables de la Tabla 4 del artículo.

In [ ]:
import time
subset_variables = ['bathrooms', 'bedrooms', 'latitude', 'longitude', 'building_type',
 'seller_type', 'total_surface', 'quality_municipality', 'parking_lots',
 'swimpool', 'terrace', 'storage_units', 'new_house', 'simce_4b',
 'nearby_schools_3km', 'dist_subway', 'dist_park', 'dist_hospital', 'dist_police',
 'dist_university', 'dist_mall', 'dist_penitentiary', 'dist_industry', 'dist_dump',
 'photo_bathroom', 'photo_bedroom', 'photo_livingroom', 'photo_diningroom',
 'photo_interior', 'photo_exterior', 'photo_balcony', 'photo_laundryroom', 'photo_floorplans', 'photo_swimpool']

X = properties[subset_variables]
y = np.log1p(properties['price'])      # variable dependiente en log1p, como en el original
training = Training(X=X, y=y, modo=MODO)

t0 = time.time()
metricas_cv = training.train_cross_validation()
print(f'Validación cruzada ({MODO}): {time.time() - t0:.0f} s')
for nombre, valor in metricas_cv.items():
    print(f'{nombre}: {valor:.4f}')

## Validación simple (80/20)
Este modelo es el que se usa para SHAP.

In [ ]:
t0 = time.time()
lgbm_model, metricas_simple, X_train, X_test, y_train, y_test, y_pred = training.train_simple_validation()
print(f'Validación simple ({MODO}): {time.time() - t0:.0f} s')
for nombre, valor in metricas_simple.items():
    print(f'{nombre}: {valor:.4f}')

## Valores SHAP

El original calcula `TreeExplainer(..., feature_perturbation='interventional', data=X_train)` sobre todo `X_train`, que es inviable en Colab. Aquí:

- `rapido`: SHAP de ruta de árbol (`tree_path_dependent`), sin fondo, sobre una muestra de filas.
- `completo`: `interventional` con un fondo de `N_FONDO_SHAP` filas (aproximación del original) sobre la misma muestra.

Los valores cambian ligeramente respecto al original, pero el ranking y las formas de las curvas se conservan.

In [ ]:
import shap
shap.initjs()

rng = np.random.default_rng(0)
n = min(N_MUESTRA_SHAP, len(X_train))
pos = rng.choice(len(X_train), size=n, replace=False)

# Dos casas para los gráficos de cascada: las del original (Las Condes y Puente Alto)
pos_alta, pos_baja = 4404, 31849
pos = np.unique(np.r_[pos, pos_alta, pos_baja])
X_shap = X_train.iloc[pos]

t0 = time.time()
if MODO == 'completo':
    fondo = X_train.sample(N_FONDO_SHAP, random_state=0)
    explainer = shap.TreeExplainer(lgbm_model, feature_perturbation='interventional', data=fondo)
else:
    explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_shap)
valor_base = float(np.ravel(explainer.expected_value)[0])
print(f'SHAP: {time.time() - t0:.0f} s sobre {len(X_shap)} filas')

### Gráficos de cascada: casa de calidad alta y casa de calidad baja

In [ ]:
# shap.waterfall_plot (versiones recientes de shap) exige un objeto shap.Explanation;
# la clase ShapObject del original ya no funciona.
for etiqueta, p in [('Calidad de comuna alta', pos_alta), ('Calidad de comuna baja', pos_baja)]:
    i = int(np.where(pos == p)[0][0])
    explicacion = shap.Explanation(values=shap_values[i], base_values=valor_base,
                                   data=X_shap.iloc[i].values, feature_names=list(X_shap.columns))
    shap.plots.waterfall(explicacion, show=False)
    plt.title(etiqueta)
    plt.show()

### Gráfico beeswarm con los valores SHAP de cada variable

In [ ]:
shap.summary_plot(shap_values, X_shap, show=False)
plt.xlabel('SHAP value (impact on property prices)')
plt.show()

### Gráficos de dependencia
1. Calidad de la comuna frente a SIMCE.
2. SIMCE frente a calidad de la comuna.
3. Superficie total frente a calidad de la comuna.
4. Número de baños frente a calidad de la comuna.
5. Colegios cercanos frente a SIMCE.

In [ ]:
fig = plt.figure(figsize=(15, 15))
gs = plt.GridSpec(3, 2, height_ratios=[1, 1, 1])
ax1, ax2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
ax3, ax4 = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
gs_last_row = gs[2, :].subgridspec(1, 4, wspace=0)
ax5 = fig.add_subplot(gs_last_row[0, 1:3])

features = ['quality_municipality', 'simce_4b', 'total_surface', 'bathrooms', 'nearby_schools_3km']
interactions = ['simce_4b', 'quality_municipality', 'quality_municipality', 'quality_municipality', 'simce_4b']
labels = ['(a)', '(b)', '(c)', '(d)', '(e)']
xjitters = [False, False, False, 0.5, False]
axes = [ax1, ax2, ax3, ax4, ax5]

for i, (feature, label, interaction, xjitter) in enumerate(zip(features, labels, interactions, xjitters)):
    shap.dependence_plot(feature, shap_values, X_shap, interaction_index=interaction, ax=axes[i],
                         show=False, x_jitter=xjitter)
    axes[i].text(0.5, -0.2, label, ha='center', va='center', transform=axes[i].transAxes, fontsize=14)
    axes[i].set_ylabel('SHAP value')
plt.tight_layout()
plt.show()

### Dependencia de las variables de distancia
Metro, parque, hospital, universidad, policía y centro comercial.

In [ ]:
features = ['dist_subway', 'dist_park', 'dist_hospital', 'dist_university', 'dist_police', 'dist_mall']
labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, (feature, label) in enumerate(zip(features, labels)):
    shap.dependence_plot(feature, shap_values, X_shap, interaction_index=None, ax=axes[i], show=False)
    axes[i].text(0.5, -0.25, label, ha='center', va='center', transform=axes[i].transAxes, fontsize=14)
    axes[i].set_ylabel('SHAP value')
plt.tight_layout()
plt.show()

### Dependencia de amenidades y fotos
Terraza, piscina, proyecto inmobiliario, foto del comedor, foto de la sala y foto de planos.

In [ ]:
X_shap_cat = X_shap.copy()
decodificacion = {0: 'No', 1: 'Yes'}
for variable in ['swimpool', 'terrace', 'new_house', 'photo_swimpool', 'photo_floorplans',
                 'photo_diningroom', 'photo_livingroom']:
    X_shap_cat[variable] = X_shap_cat[variable].map(decodificacion)

features = ['terrace', 'swimpool', 'new_house', 'photo_diningroom', 'photo_livingroom', 'photo_floorplans']
labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, (feature, label) in enumerate(zip(features, labels)):
    shap.dependence_plot(feature, shap_values, X_shap_cat, interaction_index=None, x_jitter=0.9,
                         ax=axes[i], show=False)
    axes[i].text(0.5, -0.25, label, ha='center', va='center', transform=axes[i].transAxes, fontsize=14)
    axes[i].set_ylabel('SHAP value')
plt.tight_layout()
plt.show()

## Guardar resultados

In [ ]:
os.makedirs('resultados', exist_ok=True)
pd.DataFrame({'validacion_cruzada_5': metricas_cv, 'validacion_simple_80_20': metricas_simple}).to_csv('resultados/lgbm_metricas.csv')
imp = pd.DataFrame({'variable': X_shap.columns, 'shap_abs_media': np.abs(shap_values).mean(axis=0)}).sort_values('shap_abs_media', ascending=False)
imp.to_csv('resultados/lgbm_importancia_shap.csv', index=False)
print(pd.DataFrame({'CV 5-fold': metricas_cv, 'Holdout 80/20': metricas_simple}).round(4))
print('\nModo:', MODO)
# from google.colab import files; files.download('resultados/lgbm_metricas.csv')